In [1]:
import os
import numpy as np
from pathlib import Path

# 1. 设置根目录和期望的采样点长度阈值
DATASET_ROOT = r"E:\XieYiR\rtl_ml\datasets_validated\datasets_validated"
EXPECTED_SHAPE = (512000,)  # 期望的 samples shape

def check_dataset(dataset_root, expected_shape):
    root_path = Path(dataset_root)
    if not root_path.exists():
        print(f"错误: 路径不存在 -> {dataset_root}")
        return

    total_scanned = 0
    mismatch_count = 0
    mismatched_files = []
    category_stats = {}

    print(f"开始检查数据集...")
    print(f"目标路径: {dataset_root}")
    print(f"期望 samples.shape: {expected_shape}\n" + "-" * 60)

    # 遍历每个子文件夹
    for category_dir in sorted(root_path.iterdir()):
        if not category_dir.is_dir():
            continue
        
        cat_name = category_dir.name
        category_stats[cat_name] = {"total": 0, "mismatch": 0}
        
        npy_files = list(category_dir.glob("*.npy"))
        
        for npy_file in npy_files:
            total_scanned += 1
            category_stats[cat_name]["total"] += 1
            
            try:
                # 加载 npy 数据（字典通常以 object 类型存储，需使用 .item() 提取）
                loaded_obj = np.load(npy_file, allow_pickle=True)
                data = loaded_obj.item() if hasattr(loaded_obj, 'item') and loaded_obj.ndim == 0 else loaded_obj
                
                # 检查是否存在 samples 字段
                if isinstance(data, dict) and 'samples' in data:
                    samples = data['samples']
                    current_shape = samples.shape
                    
                    # 检查 shape 是否不等于期望值
                    if current_shape != expected_shape:
                        mismatch_count += 1
                        category_stats[cat_name]["mismatch"] += 1
                        mismatched_files.append((str(npy_file), current_shape))
                else:
                    mismatch_count += 1
                    category_stats[cat_name]["mismatch"] += 1
                    mismatched_files.append((str(npy_file), "缺少 'samples' 键"))
                    
            except Exception as e:
                mismatch_count += 1
                category_stats[cat_name]["mismatch"] += 1
                mismatched_files.append((str(npy_file), f"读取错误: {str(e)}"))

    # 2. 打印分类统计结果
    print(f"{'类别名称':<20} | {'扫描总数':<10} | {'不合规数量':<10}")
    print("-" * 60)
    for cat_name, stats in category_stats.items():
        print(f"{cat_name:<20} | {stats['total']:<10} | {stats['mismatch']:<10}")
    print("-" * 60)

    # 3. 汇总与不合规详情
    print(f"\n【检查汇总】")
    print(f"总计扫描文件: {total_scanned} 个")
    print(f"不合规文件数: {mismatch_count} 个")

    if mismatch_count > 0:
        print("\n【不合规文件详情列表】:")
        for file_path, actual_shape in mismatched_files:
            print(f"文件: {file_path}")
            print(f"实际 shape / 状态: {actual_shape}\n")
    else:
        print("\n所有文件的 samples.shape 均符合要求 (512000,)！")

if __name__ == "__main__":
    check_dataset(DATASET_ROOT, EXPECTED_SHAPE)

开始检查数据集...
目标路径: E:\XieYiR\rtl_ml\datasets_validated\datasets_validated
期望 samples.shape: (512000,)
------------------------------------------------------------
类别名称                 | 扫描总数       | 不合规数量     
------------------------------------------------------------
APRS                 | 100        | 0         
FM_broadcast         | 200        | 100       
FRS_GMRS             | 100        | 0         
ISM_sensors          | 100        | 0         
NOAA_weather         | 100        | 0         
noise                | 100        | 0         
pager                | 100        | 0         
------------------------------------------------------------

【检查汇总】
总计扫描文件: 800 个
不合规文件数: 100 个

【不合规文件详情列表】:
文件: E:\XieYiR\rtl_ml\datasets_validated\datasets_validated\FM_broadcast\FM_broadcast_20260326_013845_846726.npy
实际 shape / 状态: (524288,)

文件: E:\XieYiR\rtl_ml\datasets_validated\datasets_validated\FM_broadcast\FM_broadcast_20260326_013848_448295.npy
实际 shape / 状态: (524288,)

文件: E:\XieYiR\r